# Data Quality Assessment Framework

#### Run common utils

In [25]:
%run common_utils

StatementMeta(, eff15a77-87bd-4a14-bce0-13bead9cf882, 29, Finished, Available, Finished)

#### Intialize config parametrs

In [39]:
# Define log file path
base_folder_path = '/lakehouse/default/Files/' # modify based on your environment settings
log_file_path = base_folder_path + 'logs/' + 'etl_' + datetime.now().strftime("%Y%m%d_%H%M%S") + '.log'

# Set up logger
logger = setup_logger(log_file_path)
try:
    # read, validate and parse the configuration file
    config_file_name = "etl_config.yml"
    config_file_path = base_folder_path + "config/" + config_file_name
    config, error_message = load_config(config_file_path, logger)
    validate_config(config, logger)

    # initialize config parameters
    workspace_name = config["workspace"]["workspace_name"]
    workspace_id = config["workspace"]["workspace_id"]
    metadata_db_config = config["metadata_db_config"]
    bronze_lakehouse_name = config["lakehouse_name"]["bronze"]
    silver_lakehouse_name = config["lakehouse_name"]["silver"]
    gold_lakehouse_name = config["lakehouse_name"]["gold"]
    dq_config = config["config_table_name"]["data_quality_config"]
    dq_results = config["audit_table_name"]["data_quality_audit"]
    enrich_control = config["control_table_name"]["enrich_control"]
    serve_control = config["control_table_name"]["serve_control"]

    # print(workspace_name, workspace_id, bronze_lakehouse_name, silver_lakehouse_name, enrich_control, transformation_config)

except Exception as e:
    error_msg = f"Error reading config yaml file: {str(e)}"
    logging.error(error_msg)

StatementMeta(, eff15a77-87bd-4a14-bce0-13bead9cf882, 43, Finished, Available, Finished)

Configuration loaded successfully.
Configuration validated successfully.


In [42]:
from pyspark.sql.functions import col, current_timestamp, regexp_extract
from datetime import datetime

StatementMeta(, eff15a77-87bd-4a14-bce0-13bead9cf882, 46, Finished, Available, Finished)

#### Data Quality Helper Functions

In [46]:
def perform_dq_checks(config, dataset_name, df):
    
    results = []
    for row in config:

        dq_rule_id = row["dq_rule_id"]
        control_id = row["control_id"]
        process_stage = row["process_stage"]
        dataset_name = row["dataset_name"]
        dq_rule = row["dq_rule"]
        dq_rule_criteria = row["dq_rule_criteria"] if row["dq_rule_criteria"] else None
        dq_rule_criteria_additional = row["dq_rule_criteria_additional"] if row["dq_rule_criteria_additional"] else None
        total_recs = df.count()
        if dq_rule == "duplicates_check":
            if dq_rule_criteria:
                columns_to_check = dq_rule_criteria.split(", ")
                duplicate_count = total_recs - df.dropDuplicates(columns_to_check).count()
            else:
                duplicate_count = total_recs - df.dropDuplicates(columns_to_check).count()
            recs_failed = duplicate_count
            recs_passed = total_recs - recs_failed
            dq_status = "Pass" if duplicate_count == 0 else "Fail"
            duplicate_result = f"Pass: no duplicates found" if duplicate_count == 0 else f"Fail: {duplicate_count} duplicates found"
            results.append((dq_rule_id, control_id, process_stage, dataset_name, dq_rule, dq_status, duplicate_count, duplicate_result, recs_passed, recs_failed, total_recs, dq_rule_criteria, dq_rule_criteria_additional))


        elif dq_rule == "missing_values_check":
            columns_to_check = dq_rule_criteria.split(", ")
            for column in columns_to_check:
                missing_count = df.filter(col(column).isNull()).count()
                recs_failed = missing_count
                recs_passed = total_recs - recs_failed
                dq_status = "Pass" if missing_count == 0 else "Fail"
                missing_result = "Pass: no missing values found" if missing_count == 0 else f"Fail: {missing_count} missing values in {column}"
                results.append((dq_rule_id, control_id, process_stage, dataset_name, dq_rule, dq_status, missing_count, missing_result, recs_passed, recs_failed, total_recs, dq_rule_criteria, dq_rule_criteria_additional))
   

        elif dq_rule == "invalid_values_check":
            columns_to_check = dq_rule_criteria.split(", ")
            for column in columns_to_check:
                invalid_count = df.filter(~col(column).isin(dq_rule_criteria_additional)).count()
                recs_failed = invalid_count
                recs_passed = total_recs - recs_failed
                dq_status = "Pass" if invalid_count == 0 else "Fail"
                invalid_result = "Pass: no invalid values found" if invalid_count == 0 else f"Fail: {invalid_count} invalid values in {column}"
                if dq_status == "Fail":
                    details = f"{invalid_result}. Invalid values: {df.filter(~col(column).isin(dq_rule_criteria_additional)).select(column).distinct().collect()}"
                else:
                    details = invalid_result
                results.append((dq_rule_id, control_id, process_stage, dataset_name, dq_rule, dq_status, invalid_count, details, recs_passed, recs_failed, total_recs, dq_rule_criteria, dq_rule_criteria_additional))

        elif dq_rule =="pattern_check":
            column_to_check = dq_rule_criteria
            pattern_to_check = fr"{dq_rule_criteria_additional}"

            # Extract valid data using regex and create a new column 'isValid'
            df_with_validation = df.withColumn("isValid",
                                       regexp_extract(col(column_to_check), pattern_to_check, 0))
            display(df_with_validation)

            invalid_count = df_with_validation.filter(col("isValid") == "").count()
            recs_failed = invalid_count
            recs_passed = total_recs - recs_failed
            dq_status = "Pass" if invalid_count == 0 else "Fail"
            invalid_result = "Pass: no invalid values found" if invalid_count == 0 else f"Fail: {invalid_count} invalid values in {column_to_check}"
            if dq_status == "Fail":
                details = f"""{invalid_result}. Invalid values: {df_with_validation.filter(col("isValid") == "").select(column_to_check).distinct().limit(5).collect()}"""
            else:
                details = invalid_result
            results.append((dq_rule_id, control_id, process_stage, dataset_name, dq_rule, dq_status, invalid_count, details, recs_passed, recs_failed, total_recs, dq_rule_criteria, dq_rule_criteria_additional))

        results = [tuple(list(item) if isinstance(item, set) else item for item in result) for result in results]

    return results

StatementMeta(, eff15a77-87bd-4a14-bce0-13bead9cf882, 50, Finished, Available, Finished)

In [44]:
try:
    dq_config_df = read_metadata_table(dq_config, metadata_db_config, logger)
    dataset_config = dq_config_df.filter(((dq_config_df.enable_flag) == 1)).select("control_id", "dataset_name").distinct().collect()
    dq_config = dq_config_df.filter(((dq_config_df.enable_flag) == 1)).collect()
except Exception as e:
    error_msg = f"Error reading metadata table: {str(e)}"
    logging.error(error_msg)

StatementMeta(, eff15a77-87bd-4a14-bce0-13bead9cf882, 48, Finished, Available, Finished)

Data read successfully from table: mtd.dq_config


In [45]:
for row in dataset_config:
    control_id = int(row["control_id"])
    dataset_name = row["dataset_name"]

    # Read dataset
    dataset_df = read_data_from_lakehouse("TABLE", "DELTA", dataset_name, "FULL", logger)

    filtered_dq_config = [row for row in dq_config if row["control_id"] == control_id]

    # Perform DQ checks based on configuration
    results = perform_dq_checks(filtered_dq_config, dataset_name, dataset_df)

    dq_results_schema = StructType([
        StructField("dq_rule_id", IntegerType(), False),
        StructField("control_id", IntegerType(), False),
        StructField("process_stage", StringType(), False),
        StructField("dataset_name", StringType(), False),
        StructField("dq_rule", StringType(), True),
        StructField("dq_status", StringType(), True),
        StructField("dq_result", StringType(), True),
        StructField("dq_additional_details", StringType(), True),
        StructField("recs_passed", IntegerType(), False),
        StructField("recs_failed", IntegerType(), False),
        StructField("total_recs", IntegerType(), False),
        StructField("dq_rule_criteria", StringType(), True),
        StructField("dq_rule_criteria_additional", StringType(), True)
    ])

    # Format results into a dataframe
    results_df = spark.createDataFrame(results, dq_results_schema)

    # Write results to metadata database
    write_metadata_table(results_df, dq_results, metadata_db_config, logger)

StatementMeta(, eff15a77-87bd-4a14-bce0-13bead9cf882, 49, Finished, Available, Finished)

TABLE DELTA data_lakehouse.dbo.customers FULL
PhoneNumber ^\+?1?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$ <class 'str'> PhoneNumber ^\+?1?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$


SynapseWidget(Synapse.DataFrame, 61d8d7d1-1efa-4c5d-8d06-dd756bc389b4)

Data written successfully to table: mtd.dq_results
TABLE DELTA data_lakehouse.dbo.orders FULL
Data written successfully to table: mtd.dq_results
